# D4.1 · Remediation policy — what may be done without asking

**Function D — The Agentic SOC → Respond — From a Conclusion to the Actor Stopped**

Builds on **[D3.10 · Hunting in agent telemetry](https://spbreed.github.io/cyber-commons/lessons/D3.10.html)**.

| | |
|---|---|
| Tools used | OPA |

## What this lesson is

**What it covers.** The remediation policy: classifying every response action on reversibility and blast radius, and deriving each runbook's automation tier from that rather than from its author.

**Why a security engineer needs it.** If tiers are chosen per runbook, the blast radius of your incident response is unknown until it fires. Two properties of the action decide it, and reversibility outranks radius — which is why deleting one agent's workdir is manual while forcing human-in-the-loop across the whole estate is not.

## 1 · The hook

Every runbook in the drawer picked its own automation tier, chosen by whoever wrote it on the day. So the blast radius of your incident response is unknown until the response fires, which is the worst moment to find out.

> **At CyberTravels.** The nine actions are CyberTravels' actual response levers, from throttling one agent to rotating the estate's root CA. The pair worth reading together is deleting a single agent's working directory — manual, because there is no undo — against forcing human-in-the-loop on every agent in the estate, which is one flag.

## 2 · The framework

```
                    reversible without a human?
                    yes                     no
                +---------------------+-------------------+
   one agent    |  AUTOMATED          |  MANUAL           |
                +---------------------+-------------------+
   one tenant   |  HUMAN IN THE LOOP  |  MANUAL           |
                +---------------------+-------------------+
   the estate   |  HUMAN IN THE LOOP  |  MANUAL           |
                +---------------------+-------------------+

   reversibility outranks radius, and that ordering IS the policy:
     delete one agent's workdir   one agent, no undo    -> manual
     force HITL on every agent    whole estate, a flag  -> HITL
```

If each runbook picks its own automation tier, the blast radius of your response
is unknown until the response happens.

Two properties of the **action** decide it, and neither is a property of the
person writing the runbook. Is it reversible without a human decision? And does
it touch one agent, one tenant, or the estate?

Reversibility outranks radius, and that ordering is the whole content of the
policy. Deleting one agent's working directory touches a single agent and is
still manual, because there is no undo. Forcing human-in-the-loop across the
entire estate touches everyone and is not, because it is one flag and it can be
turned back off.

> **Anchor → D1.0.** This fixes the contain interval **before** the incident, from two properties of the action rather than from whoever wrote the runbook. Chosen per runbook, the blast radius of your response is unknown until the response fires.

## 3 · Write it down once

Every runbook then cites a row rather than restating it. If you find yourself
arguing for an exception, the two properties are wrong — not the policy.

The honest test for reversibility is time. An undo that takes six hours of
manual work is not reversible for an incident that lasts twenty minutes.

## 4 · Nine actions, three tiers

The tier is derived, not assigned. Read the two estate-wide rows against the two single-agent ones.

### The skill — [`skills/response/remediation-policy-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/remediation-policy-check/SKILL.md)

```yaml
name: remediation-policy-check
description: >-
  Classify remediation actions on reversibility and blast radius so each
  runbook's automation tier is derived rather than chosen. Use before writing
  response automation, when a runbook's tier is set by its author's confidence,
  or when the blast radius of a response is unknown until it fires.
allowed-tools: Read, Grep, Glob
```

# Decide once what may be done without asking, and let every runbook inherit it

If each runbook picks its own automation tier, the blast radius of your response
is unknown until the response happens. Two properties of the *action* decide it,
and neither is a property of the person writing the runbook.

**Reversibility** — can this be undone without a human decision?
**Blast radius** — one agent, one tenant, or the estate?

Reversibility outranks radius. An irreversible action on one agent is manual; a
reversible one across the estate is human-in-the-loop.

## When to use this

Before the first runbook is written, and whenever a new remediation action is
added to the catalogue. Re-run it when an action's reversibility changes —
"revoke token" becomes irreversible the moment there is no re-issue path.

## Step-by-step

**1 — List the actions, not the incidents.** The policy is about what can be
done, not about when.

**2 — Ask whether each is reversible without a human.** An undo that needs an
approval is not reversible for this purpose.

**3 — Set the radius honestly.** "One agent" means one, not one class.

**4 — Derive the tier; do not assign it.** If you find yourself arguing for an
exception, the two properties are wrong, not the policy.

**5 — Publish the table.** Every runbook cites a row rather than restating it.

## Example

**Input** — nine actions, in
[`scripts/remediation_policy_check.py`](scripts/remediation_policy_check.py).

**Output** — the rows that make the ordering visible:

```
force HITL on every agent            True   estate   human-in-the-loop
delete the agent's workdir          False    agent   manual
```

Estate-wide and reversible is a lower tier than single-agent and irreversible.

## Output contract

```json
{
  "actions": [{"action": "str", "reversible": true,
               "radius": "agent|tenant|estate", "tier": "automated|human-in-the-loop|manual"}],
  "by_tier": {"automated": 0, "human-in-the-loop": 0, "manual": 0}
}
```

## Common edge cases

- **Reversible in theory.** If restoring takes six hours of manual work, it is
  not reversible for an incident that lasts twenty minutes.
- **Radius depends on the target.** Then it is two actions, not one.
- **An action nobody has ever taken.** It still needs a tier, before somebody
  needs it at 3am.

## Failure modes

- **Tier by confidence.** The most automated runbooks end up written by whoever
  was most sure.
- **Radius over reversibility.** Produces automated deletes because they only
  touch one agent.
- **A policy with no published table.** Then every runbook re-derives it and
  they disagree.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/remediation-policy-check/scripts/remediation_policy_check.py
SCRIPT = "skills/response/remediation-policy-check/scripts/remediation_policy_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Nine actions split three, three and three — with every irreversible action manual regardless of how small its radius is.

## Your turn

Add an action your team performs during an incident. If you cannot answer 'reversible without a human', that is the finding.

---

**Next → [D4.2 · Runbook tiers — fully automated, human in the loop, manual](https://spbreed.github.io/cyber-commons/lessons/D4.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D4.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D4.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*